# Task 1.1: Core Contribution / Architecture
**Paper**: *Ensemble of Exemplar-SVMs for Object Detection and Beyond* — Malisiewicz, Gupta & Efros (ICCV 2011)

---

## Step-by-Step Method Description

### Step 1: Feature Extraction (HOG Representation)
- **Description**: Each image window (either a training exemplar or a candidate detection) is represented using a Histogram of Oriented Gradients (HOG) descriptor. The image is divided into spatial cells, and gradient orientations in each cell are binned into a local histogram. These local histograms are concatenated into a single feature vector of dimensionality 31 × W × H, where W and H depend on the exemplar's bounding box aspect ratio.
- **Reference**: Section 3, paragraph 1 — *'each exemplar e is defined by an image with a bounding box that specifies the extent of the object within the image. We represent each exemplar using a HOG-based [4] descriptor.'*
- **Purpose**: The HOG descriptor provides a robust, local-edge-based representation of object appearance that is invariant to small spatial translations and illumination changes. This serves as the input feature vector for training each SVM.

### Step 2: Per-Exemplar SVM Training (Exemplar-SVM)
- **Description**: For each positive training exemplar $e$, a separate linear SVM is trained using $e$ as the **sole positive example** and a large pool of image windows from images that do not contain the target category as negatives. The training minimises the hinge-loss objective: $\Omega_E(w, b) = \|w\|^2 + C_1 \cdot h(w^T x_E + b) + C_2 \sum_{x \in N_E} h(-w^T x - b)$, where $h(t) = \max(0, 1-t)$ is the hinge loss, $C_1 = 0.5$ controls positive regularisation, and $C_2 = 0.01$ controls negative regularisation.
- **Reference**: Section 3, Equation for $\Omega_E$; Section 3.2 — *'C1 = 0.5 and C2 = 0.01'*; Figure 2 illustrates the per-exemplar training concept.
- **Purpose**: By training with only one positive, each SVM becomes a highly specific detector that fires on objects similar in appearance to its exemplar. The asymmetric regularisation ($C_1 \gg C_2$) ensures the single positive is heavily penalised for misclassification, while individual negatives receive a smaller penalty. This creates a detector tuned to the appearance of one particular object instance.

### Step 3: Hard Negative Mining (Iterative Refinement)
- **Description**: Training each Exemplar-SVM is computationally challenging because the negative set is vast (millions of windows from background images). Hard-negative mining is used: the SVM is initially trained on a random subset of negatives, then it is applied to the full negative set, and any negative windows that are misclassified or fall within the margin (decision function score > −1) — called *hard negatives* — are added to the active training set. The SVM is retrained with this expanded set. This process alternates for several rounds.
- **Reference**: Section 3.2 — *'We alternate between learning the weights w given an active set of negative windows N_E, and mining additional negative windows using the current w as in [9].'*
- **Purpose**: Mining ensures the decision boundary is refined against the most confusable background windows — those that look most like the exemplar. Without mining, the SVM might never see these challenging negatives and would have a loose decision boundary. This is the same strategy used in Felzenszwalb et al.'s deformable part model [9], adapted here for the per-exemplar setting.

### Step 4: Score Calibration via Platt Scaling
- **Description**: Since each Exemplar-SVM is trained independently with its own positive instance, the raw SVM decision scores from different exemplars are on different scales and are not directly comparable. To make them comparable, the authors fit a per-exemplar logistic sigmoid (Platt scaling) on a held-out validation set: $f(x | w_E, \alpha_E, \beta_E) = \frac{1}{1 + e^{-\alpha_E(w_E^T x - \beta_E)}}$. The parameters $\alpha_E$ and $\beta_E$ are learned per exemplar by fitting a logistic function to the raw SVM scores on validation set images.
- **Reference**: Section 3.1 — *'we fit a logistic function to these scores. The two scalar parameters of this function, alpha and beta, are estimated per-exemplar via maximum likelihood.'*; also references Platt [20].
- **Purpose**: Calibration ensures that the output of each Exemplar-SVM represents a comparable probability-like score, so exemplars with different intrinsic generalisation abilities produce scores on the same scale. The paper notes that *'different exemplars will offer drastically different generalization potential'* — some are clean frontal views of objects while others are heavily occluded, and calibration normalises for this variation.

### Step 5: Ensemble Detection (Non-Maximum Suppression)
- **Description**: At test time, every Exemplar-SVM is applied to every candidate window in the test image. The calibrated scores from all exemplars are collected, creating a set of scored detections across the image. Non-maximum suppression (NMS) is applied to suppress overlapping detections, keeping only the highest-scoring detection in each spatial region.
- **Reference**: Section 4.1 — *'At test time, each Exemplar-SVM creates detection windows, and we use standard non-maximum suppression to create a final, sparse set of detections per image.'*
- **Purpose**: The ensemble combines millions of individually-weak detectors into a collective decision. The calibrated scores ensure that the best-matching exemplar's detection takes priority during NMS, regardless of which specific exemplar produced it.

### Step 6: Exemplar-Based Meta-Data Transfer
- **Description**: Because each detection is associated with a specific training exemplar, any meta-data attached to that exemplar (segmentation masks, 3D geometry, viewpoint labels, action annotations) can be directly transferred to the detection without additional learning or inference. The authors demonstrate transfer for object segmentation, geometric surface estimation, and 3D viewpoint recognition.
- **Reference**: Section 5 — *'exemplar detectors naturally create an explicit association between a recognition-level output and a particular training exemplar'*; Figure 1 and Figure 7 illustrate meta-data transfer.
- **Purpose**: This is the primary practical contribution: a single unified framework handles detection, segmentation, pose estimation, and attribute transfer simultaneously, simply by leveraging the exemplar associations produced by the detector.

---

## Final Summary

This paper solves the problem of simultaneously performing object detection and transferring rich per-instance meta-data (segmentation, geometry, viewpoint) from training examples to test detections. The authors claim their Exemplar-SVM approach is better than existing category-level detectors (such as Dalal-Triggs HOG [4] and Felzenszwalb DPM [9]) because it directly links each detection to a specific training instance, enabling zero-cost meta-data transfer while achieving competitive detection accuracy through calibrated ensemble scoring.